Данный ноутбук представляет собой выполненную работу в рамках летней практики СПБГУ от VK.
    
**Тема**:   Обучение визуально-языковых моделей

**Описание**:   Исследование методов борьбы с феноменом *critical forgetting* на примере дообученной на датасете
ruCommonVQA модели от vk *llava-gemma-2b-lora*. Изначально модель дообучается с помощью механизма *QLora*(параметры см. Qlora_config), проводится сравнение метрики *rouge* на *GQA-ru* до и после дообучения. Далее исследуется эффективность механизмов дообучения VLM, разработанных для борьбы с critical forgetting: . Оригинальные статьи указаны в цитатах.

**Выполнил:** Кайманов К. студент 2 курса ПМ-ПУ СПБГУ

In [1]:

import numpy as np
import pandas as pd
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from datasets import load_dataset
import kagglehub
import logging
import torch
import PIL
from PIL import Image
from datasets import load_dataset, Dataset
from io import BytesIO

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
#задание логгера, чтобы видеть скорость выполнения скрипта. 
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/andreykaymanov/llava-gemma-2b-tuned/processor/llava-gemma-simple/tokenizer.json
/kaggle/input/datasets/andreykaymanov/llava-gemma-2b-tuned/processor/llava-gemma-simple/tokenizer_config.json
/kaggle/input/datasets/andreykaymanov/llava-gemma-2b-tuned/processor/llava-gemma-simple/chat_template.jinja
/kaggle/input/datasets/andreykaymanov/llava-gemma-2b-tuned/processor/llava-gemma-simple/processor_config.json
/kaggle/input/datasets/andreykaymanov/llava-gemma-2b-tuned/lora-adapter/llava-gemma-simple-lora/adapter_model.safetensors
/kaggle/input/datasets/andreykaymanov/llava-gemma-2b-tuned/lora-adapter/llava-gemma-simple-lora/training_args.bin
/kaggle/input/datasets/andreykaymanov/llava-gemma-2b-tuned/lora-adapter/llava-gemma-simple-lora/adapter_config.json
/kaggle/input/datasets/andreykaymanov/llava-gemma-2b-tuned/lora-adapter/llava-gemma-simple-lora/README.md
/kaggle/input/datasets/andreykaymanov/llava-gemma-2b-tuned/lora-adapter/llava-gemma-simple-lora/tokenizer.json


Препроцессинг. Подготовка датасета.

In [2]:

ru_common = "ru_common" 
gqa = "GQA"

''' Переменные для обозначения датасетов, чтобы не писать под каждый отдельную load_ds. Общий механизм подгрузки 
у них схожий. '''

' Переменные для обозначения датасетов, чтобы не писать под каждый отдельную load_ds. Общий механизм подгрузки \nу них схожий. '

In [3]:
def format_to_gemma_style(conversation: list) -> dict: #lllava chchchicken
    human = conversation[0]
    gpt = conversation[1]
    return {"conversation": {"role": "user", "content": f"<image>\n{human}"}, "answer" : gpt}

def load_image_from_url(sample):
    img = Image.open(BytesIO(sample["bytes"]))
    buffer = BytesIO()
    img.save(buffer, format='JPEG', quality=60, optimize=True) #слишком долго собирался датасет без оптимизации
    
    return Image.open(buffer)

def load_ds(dset = ru_common) -> tuple:
    if dset == ru_common:
        logger.info("загрузка данных...")
        dataset = load_dataset("MERA-evaluation/ruCommonVQA", split="test")
        logger.info("создание индекса изображений...")
        dataset_length = len(dataset["meta"])
        log_step = dataset_length // 20
        index_image = dict() #тот самый индекс
        for i in range(dataset_length):
            index_image[dataset["meta"][i]["id"]] = dataset["inputs"][i]["image"]

        logger.info("объединение, форматирование датасета под llava...")
        train_data = []
    
        for j in range(dataset_length):
            image_id = dataset["meta"][j]["id"]
            if (j + 1) % log_step == 0 or (j + 1) == dataset_length:
                percent = (j + 1) / dataset_length * 100
                logger.info(f"Прогресс: {j + 1}/{dataset_length} ({percent:.1f}%)")
            #для упрощения взят только один вариант ответа - первый.
            llava_style_instructions = format_to_gemma_style([dataset["inputs"][j]["question"], dataset["outputs"][j][0]])
            train_data.append({"image":index_image[image_id], "instruction": llava_style_instructions["conversation"], "answer": llava_style_instructions["answer"]})
            
    elif dset == gqa:
        
        logger.info("загрузка данных...")
    
        image_dataset = load_dataset("deepvk/GQA-ru", "testdev_balanced_images", streaming=True)
        text_dataset = load_dataset("deepvk/GQA-ru", "testdev_balanced_instructions", streaming=True)

        image_dataset = image_dataset["testdev"]
        text_dataset = text_dataset["testdev"]
    

        logger.info("объединение, форматирование датасета под llava...")
    
        train_data = []
        for text in text_dataset:
            image_id = text["imageId"]
            train_data.append({"image":image_id, "instruction": format_to_gemma_style([text["question"], text["answer"]])})


    dataset = Dataset.from_list(train_data)
    alpha = 0.9
    split_idx = int(len(train_data) * alpha)
    train_ds = dataset.select(range(split_idx))
    eval_ds = dataset.select(range(split_idx, len(dataset)))
    logger.info("Готово!")
    if dset == ru_common:
        return train_ds, eval_ds
    elif dset == gqa:
        return train_ds, eval_ds, image_dataset
    
#load_ds()

Создание data collator. Загрузка модели.

In [4]:
class DataCollator:
    def __init__(self, processor):
        self.processor = processor
        self.max_seq_length = 1024 #простите, лень замерять точно

    
    def __call__(self, batch):
        
        prompts = []
        answers = []
        images = []
        
        for element in batch:
            element["image"] = PIL.Image.open(BytesIO(element["image"]["bytes"])).convert("RGB")
            prompts.append(element["instruction"]["content"])
            answers.append(element["answer"])
            images.append(element["image"])

        prompt_tokens = self.processor.tokenizer(prompts, padding=False, add_special_tokens=False, return_tensors=None)
        prompt_lengths = [len(tokens) for tokens in prompt_tokens['input_ids']]

        full_text = [p + a for p, a in zip(prompts, answers)]

        inputs = self.processor(text = full_text, images = images, padding = True, truncation=True, max_length = self.max_seq_length, return_tensors="pt")

        labels = inputs['input_ids'].clone()
        for i, prompt_len in enumerate(prompt_lengths):
            labels[i, :prompt_len] = -100 #скрытие промпта, чтобы кросс-энтропия его не учитывала
        
        '''device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        for k, v in inputs.items():
            if isinstance(v, torch.Tensor):
                inputs[k] = v.to(device)        
        inputs['labels'] = labels.to(device)'''
        inputs['labels'] = labels

        return inputs

  Теперь, когда есть готовый механизм для подгрузки датасета и коллатор данных, можем дообучить модель на ruCommonVQA. Однако, для начала нужно проверить rouge as-is модели на GQA-ru test. Если метрика после tuning-а просядет, это покажет наличие феномена critical forgetting.

In [ ]:
!rm -rf /opt/conda/lib/python3.10/site-packages/aiohttp* && pip install aiohttp trl 
!pip install evaluate
!pip install bitsandbytes

In [ ]:
import evaluate
import torch
from transformers import AutoModel
from transformers import AutoProcessor, AutoTokenizer
from transformers import enable_full_determinism
from transformers import BitsAndBytesConfig
from transformers import AutoModelForMultimodalLM
rouge = evaluate.load("rouge") #фактически метрика анализирует вхождение слов в ответ или полное совпадение.
model_name = "deepvk/llava-gemma-2b-lora" #использованы материалы вк
enable_full_determinism(42)
'''bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForMultimodalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="balanced",
    trust_remote_code=True,
)'''
tokenizer = AutoTokenizer.from_pretrained(model_name)
processor = AutoProcessor.from_pretrained(
    model_name,
    patch_size=14,
    trust_remote_code=True,
)
processor.image_processor.size = {"shortest_edge": 224, "longest_edge": 224}

In [ ]:
gqa_dataset = load_ds(gqa)

In [7]:
import time
from IPython.display import Image, display
import io
import re
image_dataset = gqa_dataset[2]
    
#model.eval()


def normalize(text):
    text = re.sub(r'[^\w\s]', '', text)
    return text.lower().strip()


def get_image(image_id):
    for image in image_dataset:
            if image_id == image["id"]:
                return image["image"]
                

def predict(model, processor, image, prompt, tokenizer):
    #Генерация ответа модели
    
    
    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    )
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.cuda.is_available():
        print(f"Имя GPU: {torch.cuda.get_device_name(0)}")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.to(device)
    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=7,          
            do_sample=False,           
        )
    
    generated_text = processor.decode(
        outputs[0][inputs['input_ids'].shape[1]:], 
        skip_special_tokens=True
    )
    del inputs, outputs
    torch.cuda.empty_cache()
    
    return generated_text


def predict_batch(model, processor, images, prompts, max_new_tokens=10):
    texts = []
    for i in range(len(prompts)):
         text = tokenizer.apply_chat_template([prompts[i]], tokenize=False, add_generation_prompt=True)
         texts.append(text)

    
    inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id
        )
    
    responses = []
    input_length = inputs['input_ids'].shape[1]
    for i in range(len(images)):
        response = processor.decode(
            outputs[i][input_length:],
            skip_special_tokens=True
        )
        responses.append(response)
    del inputs, outputs
    torch.cuda.empty_cache()
    
    return responses


predicted_answers = []
true_answers = []
short_phrase = " Ответь кратко."
gqa_train = gqa_dataset[1]
print(gqa_train)
ln = len(gqa_train)
'''for element in gqa_train:
    i += 1
    image = get_image(element["image"])
    conversation = str(element["instruction"])
    answer_index = conversation.find("ASSISTANT: ") + len("ASSISTANT: ")
    prompt = conversation[:answer_index]
    answer = conversation[answer_index:]
    print(image)
    print(prompt)
    start = time.time()
    predicted_answer = predict(model, processor, image, prompt)
    end = time.time()
    print("ВРЕМЯ:", end - start)
    predicted_answers.append(predicted_answer)
    true_answers.append(answer)'''

def get_rouge(batch_size, ln, model, processor):
    for i in range(0, ln, batch_size):
        batch_prompts = []
        batch_images = []
        batch_true = []
        for j in range(i, min(i + batch_size, ln)):
            image = get_image(gqa_train["image"][j])
            
            please_shorter = gqa_train["instruction"][j]["conversation"]
            please_shorter["content"] += short_phrase
            prompt =  please_shorter
            
            true_answer = gqa_train["instruction"][j]["answer"]
            #print(prompt, true_answer)
            #buffered = io.BytesIO()
            #image.save(buffered, format="PNG")
            #display(Image(data=buffered.getvalue()))
            batch_prompts.append(prompt)
            batch_images.append(image)
            batch_true.append(true_answer)
        
        print(str(i / ln) + "%", "обработано")
        batch_predictions = predict_batch(model, processor, batch_images, batch_prompts)
        #print(batch_predictions)
        batch_predictions = [p.lstrip('\n') for p in batch_predictions]
        predicted_answers.extend(batch_predictions)
        true_answers.extend(batch_true)

    
    true_norm = [normalize(t) for t in true_answers]
    pred_norm = [normalize(p) for p in predicted_answers]
    for i, (p, r) in enumerate(zip(pred_norm, true_norm)):
        print(f"Pair {i}: pred='{p}', ref='{r}'")
        
    rouge_asis = rouge.compute(
        predictions=pred_norm,
        references=true_norm,
        tokenizer = lambda x: x.split() #rouge, как оказывается, из коробки слабо поддерживает русский
        )
    return rouge_asis

#get_rouge(3, len(gqa_train), model)   

Dataset({
    features: ['image', 'instruction'],
    num_rows: 1222
})


rouge1, rougeLsum: **0.44**. Скорее всего, падение метрики относительно представленной в материалах вк связано с использованием четырехбитной квантизации и недостаточно отлаженного пайплайна обработки промптов, однако в пределах ожидаемого. Можно приступать к дообучению модели.

In [ ]:

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

train_ds, eval_ds = load_ds()


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = prepare_model_for_kbit_training(model)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="./llava-gemma-lora-sft",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    dataloader_num_workers = 2,
    bf16 = False,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    report_to=[],
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollator(processor),
    processing_class=processor.tokenizer,
)

trainer.train()

trainer.save_model("./llava-gemma-simple-lora")
processor.save_pretrained("./llava-gemma-simple")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


trainable params: 21,970,944 || all params: 2,838,077,440 || trainable%: 0.7741


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
50,13.029514,12.779972


In [10]:
import zipfile
import os

def zip_folder(folder_path, zip_name):
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                zipf.write(os.path.join(root, file),
                           os.path.relpath(os.path.join(root, file),
                                           os.path.join(folder_path, '..')))

zip_folder('./llava-gemma-simple-lora', 'lora-adapter.zip')
zip_folder('./llava-gemma-simple', 'processor.zip')
#Сохранение адаптера и процессора для дальнейшего создания kaggle датасет.

In [ ]:
import zipfile
import os

zip_path = "/kaggle/input/llava-gemma-2b_tuned/lora-adapter.zip"

extract_path = "/kaggle/working/llava-gemma-simple-lora"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

#распаковка зип-архива с Lora

In [ ]:
from transformers import AutoModelForMultimodalLM, BitsAndBytesConfig
from peft import PeftModel
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
base_model = AutoModelForMultimodalLM.from_pretrained(
    "deepvk/llava-gemma-2b-lora",
    device_map="auto",
    quantization_config = bnb_config,
    torch_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained("deepvk/llava-gemma-2b-lora")
processor.image_processor.patch_size = 14


#использовать для инференса

In [ ]:
!pip install --upgrade torchao peft #по умолчанию в kaggle стоит 0.10.0, что peft-у очень не понравилось

In [11]:
path = "/kaggle/input/datasets/andreykaymanov/llava-gemma-2b-tuned/lora-adapter/llava-gemma-simple-lora"

if os.path.exists(path):
    print("Файлы в папке:", os.listdir(path))
else:
    print("no")
model = PeftModel.from_pretrained(
    base_model,
    path 
)

model.eval()

Файлы в папке: ['adapter_model.safetensors', 'training_args.bin', 'adapter_config.json', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja']


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlavaForConditionalGeneration(
      (model): LlavaModel(
        (vision_tower): CLIPVisionModel(
          (vision_model): CLIPVisionTransformer(
            (embeddings): CLIPVisionEmbeddings(
              (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
              (position_embedding): Embedding(577, 1024)
            )
            (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (encoder): CLIPEncoder(
              (layers): ModuleList(
                (0-23): 24 x CLIPEncoderLayer(
                  (self_attn): CLIPAttention(
                    (k_proj): lora.Linear4bit(
                      (base_layer): Linear4bit(in_features=1024, out_features=1024, bias=True)
                      (lora_dropout): ModuleDict(
                        (default): Dropout(p=0.05, inplace=False)
                      )
                      (lora_A): ModuleDic

Теперь, когда модель успешно дообучена, посчитаем rouge еще раз

In [16]:
from transformers import AutoProcessor
model_name = ""
processor = AutoProcessor.from_pretrained(
    'deepvk/llava-gemma-2b-lora',
    patch_size=14,
    trust_remote_code=True,
)

after_learning_rouge = get_rouge(4, ln, model, processor) #вместо 10 подсавить длину датасета для оценки(1223)
after_learning_rouge

2026-07-15 20:18:28,357 - INFO - HTTP Request: HEAD https://huggingface.co/deepvk/llava-gemma-2b-lora/resolve/main/processor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-15 20:18:28,375 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/deepvk/llava-gemma-2b-lora/30f1c8d674b580fcbfcdc6b1479a1f95c429b24a/processor_config.json "HTTP/1.1 200 OK"
2026-07-15 20:18:28,422 - INFO - HTTP Request: GET https://huggingface.co/api/models/deepvk/llava-gemma-2b-lora/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-07-15 20:18:28,465 - INFO - HTTP Request: HEAD https://huggingface.co/deepvk/llava-gemma-2b-lora/resolve/main/processor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-15 20:18:28,485 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/deepvk/llava-gemma-2b-lora/30f1c8d674b580fcbfcdc6b1479a1f95c429b24a/processor_config.json "HTTP/1.1 200 OK"
2026-07-15 20:18:28,541 - INFO - HTTP Re

0.0% обработано


2026-07-15 20:18:50,170 - INFO - HTTP Request: GET https://huggingface.co/datasets/deepvk/GQA-ru/resolve/e054665bda99b465da5617334727c103cd245a57/testdev_balanced_images/testdev-00000-of-00001.parquet "HTTP/1.1 302 Found"
2026-07-15 20:18:50,227 - INFO - HTTP Request: GET https://cas-bridge.xethub.hf.co/xet-bridge-us/6627a6e03d64429f459f42e5/43e4144d295ac64bae272bdbfeba5dd57fbc588839cec70a6b9c80d39c366f86?Expires=1784150330&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY2MjdhNmUwM2Q2NDQyOWY0NTlmNDJlNS80M2U0MTQ0ZDI5NWFjNjRiYWUyNzJiZGJmZWJhNWRkNTdmYmM1ODg4MzljZWM3MGE2YjljODBkMzljMzY2Zjg2KiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NDE1MDMzMH19fV19&Signature=MEYCIQDvIUww4uDO522AHq6B2Ye6d3ai-vaY0SXFJvTWX0JFKwIhAJvtHH6IfsciFp4rY6N3%7EMSM9cMDoIM9DEFTKjEVrZPg&Key-Pair-Id=K3EPXBYC3CKDRZ&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27testdev-00000-of-00001.parquet%3B+filename%3D

0.0032733224222585926% обработано


2026-07-15 20:19:03,596 - INFO - HTTP Request: GET https://huggingface.co/datasets/deepvk/GQA-ru/resolve/e054665bda99b465da5617334727c103cd245a57/testdev_balanced_images/testdev-00000-of-00001.parquet "HTTP/1.1 302 Found"
2026-07-15 20:19:03,642 - INFO - HTTP Request: GET https://cas-bridge.xethub.hf.co/xet-bridge-us/6627a6e03d64429f459f42e5/43e4144d295ac64bae272bdbfeba5dd57fbc588839cec70a6b9c80d39c366f86?Expires=1784150343&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY2MjdhNmUwM2Q2NDQyOWY0NTlmNDJlNS80M2U0MTQ0ZDI5NWFjNjRiYWUyNzJiZGJmZWJhNWRkNTdmYmM1ODg4MzljZWM3MGE2YjljODBkMzljMzY2Zjg2KiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NDE1MDM0M319fV19&Signature=MEYCIQCC3XalWJ55rMw9BfD4Dcfhi4hfR2fPNoYC2JqI7KPCcgIhAO2h6UMZ4B7pdkNvECmFkeTdUNjVlHgHB7plohYsjA6S&Key-Pair-Id=K3EPXBYC3CKDRZ&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27testdev-00000-of-00001.parquet%3B+filename%3D%2

...

2026-07-15 21:22:58,799 - INFO - HTTP Request: GET https://huggingface.co/datasets/deepvk/GQA-ru/resolve/e054665bda99b465da5617334727c103cd245a57/testdev_balanced_images/testdev-00000-of-00001.parquet "HTTP/1.1 302 Found"
2026-07-15 21:22:58,855 - INFO - HTTP Request: GET https://cas-bridge.xethub.hf.co/xet-bridge-us/6627a6e03d64429f459f42e5/43e4144d295ac64bae272bdbfeba5dd57fbc588839cec70a6b9c80d39c366f86?Expires=1784154178&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY2MjdhNmUwM2Q2NDQyOWY0NTlmNDJlNS80M2U0MTQ0ZDI5NWFjNjRiYWUyNzJiZGJmZWJhNWRkNTdmYmM1ODg4MzljZWM3MGE2YjljODBkMzljMzY2Zjg2KiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NDE1NDE3OH19fV19&Signature=MEUCIGuRNQvjbzCmpeseHdyPhAg-4JfcX9SVlRz1yYDQeEhJAiEAivZUIUbvcefUYgVdwHvLdU1SR8GJBVgCsUCadtE4gtg_&Key-Pair-Id=K3EPXBYC3CKDRZ&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27testdev-00000-of-00001.parquet%3B+filename%3D%2

0.9950900163666121% обработано


2026-07-15 21:23:09,754 - INFO - HTTP Request: GET https://huggingface.co/datasets/deepvk/GQA-ru/resolve/e054665bda99b465da5617334727c103cd245a57/testdev_balanced_images/testdev-00000-of-00001.parquet "HTTP/1.1 302 Found"
2026-07-15 21:23:09,798 - INFO - HTTP Request: GET https://cas-bridge.xethub.hf.co/xet-bridge-us/6627a6e03d64429f459f42e5/43e4144d295ac64bae272bdbfeba5dd57fbc588839cec70a6b9c80d39c366f86?Expires=1784154189&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY2MjdhNmUwM2Q2NDQyOWY0NTlmNDJlNS80M2U0MTQ0ZDI5NWFjNjRiYWUyNzJiZGJmZWJhNWRkNTdmYmM1ODg4MzljZWM3MGE2YjljODBkMzljMzY2Zjg2KiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NDE1NDE4OX19fV19&Signature=MEUCIEmeZO%7Eo0C5N9lsa-mmM1AHF8G9ZJNnLQL1RvzvEEdilAiEA3W8yDqRGJvIkiaeTFd42uyU7zipmMXAhlfD93%7EYq4cU_&Key-Pair-Id=K3EPXBYC3CKDRZ&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27testdev-00000-of-00001.parquet%3B+filename%3D%22testdev-00000-of

0.9983633387888707% обработано
Pair 0: pred='нет', ref='нет'
Pair 1: pred='нет', ref='нет'
Pair 2: pred='нет', ref='да'
Pair 3: pred='металл', ref='из металла'
Pair 4: pred='дерево', ref='металл'
Pair 5: pred='справа', ref='справа'
Pair 6: pred='да', ref='да'
Pair 7: pred='да', ref='да'
Pair 8: pred='длинным', ref='с коротким'
Pair 9: pred='окно', ref='дверь'
Pair 10: pred='на столе', ref='разделочная доска'
Pair 11: pred='картофель', ref='морковь'
Pair 12: pred='пластик', ref='из пластика'
Pair 13: pred='нет', ref='нет'
Pair 14: pred='перец', ref='яблоки'
Pair 15: pred='мужчина', ref='мальчик'
Pair 16: pred='cònслева', ref='справа'
Pair 17: pred='картофель', ref='хотдоги'
Pair 18: pred='курица', ref='бекон'
Pair 19: pred='мясо', ref='бекон'
Pair 20: pred='курица', ref='бекон'
Pair 21: pred='нет', ref='нет'
Pair 22: pred='ландgrüße', ref='лес'
Pair 23: pred='нет', ref='да'
Pair 24: pred='красного', ref='серый'
Pair 25: pred='стол', ref='письменный стол'
Pair 26: pred='стол', ref='письм

{'rouge1': np.float64(0.34056192034915456),
 'rouge2': np.float64(0.010365521003818875),
 'rougeL': np.float64(0.34068466993998914),
 'rougeLsum': np.float64(0.34026186579378076)}

rouge1, rougeLsum: **0.34** - проседание на 10 п.п. Результаты эксперимента показали, что QLora не является панацеей для critical forgetting. Стоит, однако, отметить, что датасеты имели выразительно разные характеры: GQA-ru ориентриуется на односложный ответ, в то время, как ruCommonVQA походит на llava instruct и ориентируется на сравнительно длинные ответы. Это повлияло на итоговые значения, так как модель для уменьшения loss-а в QLora "училась" отвечать развернуто.

Возможные дальнейшие направления исследований: реализовать L2-Lora, чтобы оценить потенциал этой техники в применении к легковесным моделям.
Статья: https://ojs.aaai.org/index.php/AAAI/article/view/40784

